In [ ]:
import os
import re
import time
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    import ftfy  # canonical mojibake fixer; pip install ftfy
except ImportError:
    ftfy = None


def clean_text(s: str) -> str:
    """Repair encoding damage and tidy whitespace, but keep the article's REAL punctuation
    (curly quotes, en/em dashes) so downstream character offsets stay faithful to the source.
    This is the upstream fix that makes fix_mojibake()/_demojibake() elsewhere a safety net."""
    if not s:
        return s
    # 1. repair UTF-8-read-as-Latin-1 damage ("Â£" -> "£", "â€“" -> "–")
    if ftfy is not None:
        s = ftfy.fix_text(s)
    elif any(m in s for m in ("Ã", "Â", "â€")):
        try:
            s = s.encode("latin-1", "ignore").decode("utf-8", "ignore")
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
    # 2. canonical unicode form + strip invisible junk (NBSP, zero-width, soft hyphen)
    s = unicodedata.normalize("NFC", s)
    s = s.replace(" ", " ").replace("​", "").replace("­", "")
    # 3. close the gap the get_text(" ") separator leaves before punctuation, inside brackets
    #    and before clitics ("Hormuz ." -> "Hormuz.", "( WWA )" -> "(WWA)", "Iran 's" -> "Iran's")
    s = re.sub(r"[ \t]+([’'](?:s|t|re|ve|ll|d|m)\b)", r"\1", s)
    s = re.sub(r"[ \t]+([,.;:!?%])", r"\1", s)
    s = re.sub(r"([(\[“])[ \t]+", r"\1", s)
    s = re.sub(r"[ \t]+([)\]”])", r"\1", s)
    # 4. tidy whitespace without touching real newlines between paragraphs
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


# --- Fetch strategy ---------------------------------------------------------------------------
# cbsnews.com (Akamai) bot-scores the caller and starts answering 406 to *every* request once it
# has flagged you — including ones that succeeded moments earlier. It is rate-driven, not header-
# driven, so the fix is: reuse one Session (keeps the clearance cookie), pause between articles,
# and back off on a challenge. Dropping the User-Agent entirely is the variant that reliably
# clears an active 406 block, so it is kept as the last-resort retry.
_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
_ACCEPT = "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"

HEADER_VARIANTS = [
    {"User-Agent": _UA, "Accept": _ACCEPT, "Accept-Language": "en-US,en;q=0.9",
     "Upgrade-Insecure-Requests": "1"},
    # no User-Agent at all: an Akamai edge that is 406-ing the browser UA still serves this
    {"User-Agent": None, "Accept": _ACCEPT, "Accept-Language": "en-US,en;q=0.9"},
]

CHALLENGE_CODES = {403, 406, 429, 503}


def fetch(url, session=None, attempts=3, base_wait=5.0):
    """GET `url`, retrying across header variants with exponential backoff.

    Returns the successful Response. Raises the last error if every attempt fails."""
    session = session or requests.Session()
    last_exc = None
    for attempt in range(attempts):
        for headers in HEADER_VARIANTS:
            try:
                r = session.get(url, headers=headers, timeout=30)
                if r.status_code in CHALLENGE_CODES:
                    last_exc = requests.HTTPError(
                        f"{r.status_code} challenge for url: {url}", response=r
                    )
                    continue
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                last_exc = e
        wait = base_wait * (2 ** attempt)
        print(f"      … blocked, backing off {wait:.0f}s before retry {attempt + 2}/{attempts}")
        time.sleep(wait)
    raise last_exc


# --- Body extraction --------------------------------------------------------------------------
# Taking every <p> on the page pulls in site chrome AND, on BBC, the "Read next" teasers for
# unrelated articles — plain prose that no keyword filter can distinguish from body text and that
# would otherwise contribute perfectly plausible triples about the wrong story. So scope to the
# article container first; the blocklist below only mops up chrome that lives inside it.
# Order matters: site-specific classes come before the generic tags, because CBS pages carry a
# dozen <article> elements (one per teaser card) while .content__body is unique to the story.
BODY_SELECTORS = [
    "[itemprop='articleBody']",   # CNN
    ".wysiwyg--all-content",      # Al Jazeera
    ".t-content__body",           # France 24
    ".content__body",             # CBS News
    ".article__content",
    "article",                    # DW, BBC
    "main",
]

DROP_TAGS = ["figcaption", "aside", "nav", "footer", "header", "form", "script", "style"]

BOILERPLATE = re.compile(r"""^(
      watch\ cbs\ news | headings | save | share | in: | read\ next
    | today's\ top\ stories | most\ read | follow\ al\ jazeera.* | \(\d+:\d+\)
    | .*©.*all\ rights\ reserved.* | copyright\ \d{4}\ bbc.* | .*scan\ the\ qr\ code.*
    | \(france\ 24\ with\ afp\) | .*please\ enable\ javascript.* | .*enable\ advertisement\ tracking.*
    | .*browser\ extensions\ seems\ to\ be\ blocking.* | this\ video\ can\ not\ be\ played
    | edited\ by:.* | .*don't\ let\ the\ algorithm\ hide\ the\ news.* | issued\ on:.*
    | .*sign\ up\ for\ our.*newsletter.* | hot\ stocks | markets | fear\ &\ greed\ index
    | latest\ market\ news | .*most\ stock\ quote\ data\ provided.* | --
)$""", re.I | re.X)

MIN_BODY_CHARS = 800  # below this the page served a consent/JS wall, not an article


def extract_body(html):
    """Article paragraphs, in order, with chrome removed."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(DROP_TAGS):
        tag.decompose()

    root = None
    for selector in BODY_SELECTORS:
        nodes = soup.select(selector)
        if len(nodes) == 1 and len(nodes[0].find_all("p")) >= 3:
            root = nodes[0]
            break

    paragraphs = []
    for p in (root or soup).find_all("p"):
        # The " " separator is load-bearing: get_text(strip=True) concatenates a paragraph's
        # inline children with nothing between them, and since <a> wraps entity names that
        # silently welds them to their neighbours ("rockedIranandKuwaiton Sunday").
        text = clean_text(p.get_text(" ", strip=True))
        if text and not BOILERPLATE.match(text):
            paragraphs.append(text)
    return paragraphs


def get_article_text(url, session=None):
    response = fetch(url, session=session)

    # Pass raw BYTES (not response.text): lets BeautifulSoup sniff the page's declared charset
    # instead of requests defaulting to ISO-8859-1 when the header omits it — that default is the
    # source of the "Â£3.74m" mojibake we were repairing downstream.
    article_text = "\n".join(extract_body(response.content))

    if len(article_text) < MIN_BODY_CHARS:
        # Le Monde answered 200 with "Please enable JavaScript to proceed." and the old scraper
        # saved that 36-character string as a normal article. Fail loudly instead.
        raise ValueError(
            f"body too short ({len(article_text)} chars) — probably a JS/consent wall: "
            f"{article_text[:60]!r}"
        )
    return article_text


def _existing_title(path):
    """First-line Title: of an already-scraped file, or None."""
    try:
        with open(path, encoding="utf-8") as f:
            first = f.readline()
    except OSError:
        return None
    return first[len("Title:"):].strip() if first.startswith("Title:") else None


# --- Which rows to scrape ---------------------------------------------------------------------
# 1-based and inclusive, matching the artN numbering exactly (row 1 -> articles/art1.txt) and the
# [n/total] progress log. After APPENDING new rows to articles.csv, set START_ROW to the first new
# row so a top-up run cannot refetch — or refail — anything already on disk: some sources that were
# reachable when first scraped now block us, and skip_existing alone still costs one request per row.
# END_ROW = None means "to the last row".
#
# Only ever APPEND to articles.csv. artN is the row POSITION, so inserting or reordering a row
# renumbers every file after it while results/<model>/artN/ keeps pointing at the old text.
START_ROW = 1
END_ROW = None


def main(skip_existing=True, delay=3.0, start_row=None, end_row=None):
    """Scrape rows START_ROW..END_ROW of articles.csv into articles/art{i+1}.txt.

    start_row / end_row override the module-level constants for a one-off run without editing them.

    skip_existing=True leaves a file alone when its Title header already matches the CSV row,
    so a re-run only refetches what is missing or stale. This matters because some sources
    (usnews.com) that were reachable when first scraped now block us outright — refetching
    them unconditionally would trade good text for a failure. Pass skip_existing=False to
    re-extract the whole corpus after changing the extraction rules above."""
    df = pd.read_csv("articles.csv")

    os.makedirs("articles", exist_ok=True)

    total = len(df)
    first = START_ROW if start_row is None else start_row
    last = END_ROW if end_row is None else end_row
    last = total if last is None else last
    if not 1 <= first <= last <= total:
        raise ValueError(f"row range {first}..{last} is outside articles.csv (1..{total})")
    # .iloc keeps the ORIGINAL index, so art{i+1} stays tied to the row's position in the full CSV
    # even when only a slice is being scraped.
    rows = df.iloc[first - 1:last]
    print(f"rows {first}-{last} of {total} ({len(rows)} article(s))\n")

    session = requests.Session()
    failed = []

    for i, row in rows.iterrows():
        title = row["title"]
        url = row["url"]
        date = row.get("Date", "")

        output_file = os.path.join("articles", f"art{i+1}.txt")

        print(f"[{i+1}/{total}] Processing: {title}")

        if skip_existing and _existing_title(output_file) == clean_text(title):
            print(f"    · Up to date, skipped ({output_file})")
            continue

        try:
            article_text = get_article_text(url, session=session)

            with open(output_file, "w", encoding="utf-8") as f:
                f.write(f"Title: {clean_text(title)}\n")
                f.write(f"URL: {url}\n")
                f.write(f"Date: {date}\n")
                f.write("\n")
                f.write(article_text)

            print(f"    ✓ Saved to {output_file} ({len(article_text)} characters)")

        except Exception as e:
            print(f"    ✗ Failed: {e}")
            failed.append((i + 1, title, str(e)))

        time.sleep(delay)  # stay under the rate threshold that triggers the 406 wall

    print("\nDone!")
    if failed:
        print(f"\n{len(failed)} row(s) failed — existing files for them were left untouched:")
        for n, t, e in failed:
            print(f"  row {n}: {t[:60]} — {e[:80]}")


if __name__ == "__main__":
    main()

[1/79] Processing: France records 2,025 excess deaths at peak of heatwave as Europe braces for more extreme weather
    ✓ Saved to articles/art1.txt (3744 characters)
[2/79] Processing: Taylor Swift's rumoured wedding celebrations begin with star-studded New York event
    ✓ Saved to articles/art2.txt (6668 characters)
[3/79] Processing: Plots, love letters and remedies: The medieval secrets being revealed by AI
    ✓ Saved to articles/art3.txt (11310 characters)
[4/79] Processing: What is US Open prize money?
    ✓ Saved to articles/art4.txt (1359 characters)
[5/79] Processing: Ebola treatments trial begins in DR Congo
    ✓ Saved to articles/art5.txt (3806 characters)
[6/79] Processing: Nagelsmann quits as Germany coach after early World Cup 2026 exit
    ✓ Saved to articles/art6.txt (2637 characters)
[7/79] Processing: Israel approves plan to establish 13 new settlements in occupied West Bank
    ✓ Saved to articles/art7.txt (3053 characters)
[8/79] Processing: ‘Never the same’: How